# SignSage T5-small Pose-to-Text POC (Kaggle GPU)

**Nothing in this notebook runs unless `EXECUTE = True`.** Each phase is a separate gated cell; run them in order.

Hard rules: never touch `research/artifacts/352868924`, never modify serving code, never skip the gate.

## PHASE 3 IS A HARD GO/NO-GO BOUNDARY
- **PHASE 3 PASS (>=95% EM on 64 real samples)** -> proceed to PHASE 4 (5k).
- **PHASE 3 FAIL** -> STOP. Diagnose, do not start PHASE 4. A failed gate is a result, not a cue to train longer.

In [ ]:
# Master switch. Flip to True only to run a phase, one cell at a time.
EXECUTE = False
REPO = "/kaggle/working/signsage"  # clone location of this repo
POC = f"{REPO}/research/t5_pose_poc"
OUT = "/kaggle/working/t5_poc"     # ephemeral; persist via PHASE 6
print("EXECUTE =", EXECUTE)

## PHASE 0 — environment / setup (needs HF_TOKEN with iSign access for PHASE 2+)

In [ ]:
assert EXECUTE, "set EXECUTE = True to run this phase"
%cd {REPO}
!pip install -q transformers huggingface_hub pose-format jiwer sacrebleu rouge_score nltk
!python -c "import torch, transformers; print(torch.__version__, torch.cuda.is_available(), transformers.__version__)"
# T5-small is public; pre-cache weights + tokenizer now:
!python -c "from transformers import T5Tokenizer, T5ForConditionalGeneration; T5Tokenizer.from_pretrained('google-t5/t5-small'); T5ForConditionalGeneration.from_pretrained('google-t5/t5-small'); print('t5-small cached')"

## PHASE 1 — materialization smoke / resume verification (no dataset needed)

In [ ]:
assert EXECUTE, "set EXECUTE = True to run this phase"
!python {POC}/materialize.py --smoke --out {OUT}/smoke --shard-size 10 --sanity
# Prove resume: second run must skip everything and stay valid.
!python {POC}/materialize.py --smoke --out {OUT}/smoke --shard-size 10
# Prove interruption recovery: delete the last shard, re-run, check sha256 restore.
!python -c "import json, pathlib; m = json.load(open('{OUT}/smoke/train_manifest.json')); print('shards:', len(m['shards']), 'skip_rate:', m['skip_rate'])"

## PHASE 2 — real-data materialization (~5k, one streaming pass; resumable on interrupt — just re-run the same command)

In [ ]:
assert EXECUTE, "set EXECUTE = True to run this phase"
# HF_TOKEN must be set (Kaggle Secrets) with access to Exploration-Lab/iSign.
# CSV: attach iSign_v1.1.csv as a Kaggle dataset input and point --csv at it.
!python {POC}/materialize.py --csv /kaggle/input/isign/isign_v1.1.csv --pose-source remote \
    --out {OUT}/shards --target-train 4000 --target-val 500 --target-test 500 --sanity
# On preemption: re-run the identical command; valid shards are skipped via sha256.

## PHASE 3 — real 64-sample overfit gate (BINDING). PASS >= 95% EM or STOP here.

In [ ]:
assert EXECUTE, "set EXECUTE = True to run this phase"
!python {POC}/train.py --real-gate --shards {OUT}/shards --out {OUT}/gate64 --gate-n 64
!cat {OUT}/gate64/gate_report.json

## PHASE 4 — 5k experiment (ONLY if PHASE 3 passed). Staged A (frozen) then B (joint), val-tracked, resumable.

In [ ]:
assert EXECUTE, "set EXECUTE = True to run this phase"
# Gate must show passed:true before running this cell. Resume after preemption with --resume.
!python {POC}/train.py --shards {OUT}/shards --out {OUT}/exp5k --epochs 8 --batch 16 --patience 3
# !python {POC}/train.py --shards {OUT}/shards --out {OUT}/exp5k --resume  # after interruption

## PHASE 5 — evaluation (WER / BLEU-1..4 / ROUGE-L / EM / diversity / baseline / sensitivity / 30 examples)

In [ ]:
assert EXECUTE, "set EXECUTE = True to run this phase"
!python {POC}/evaluate.py --checkpoint {OUT}/exp5k/best.pt --shards {OUT}/shards --out {OUT}/eval5k

## PHASE 6 — artifact persistence (never rely on `/kaggle/working` surviving). Publish shards + checkpoints + reports as a versioned Kaggle dataset.

In [ ]:
assert EXECUTE, "set EXECUTE = True to run this phase"
!ls -R {OUT}/shards | head -20
!ls -lh {OUT}/exp5k/ {OUT}/eval5k/ {OUT}/gate64/
# Then: Kaggle UI -> Output -> ... OR `kaggle datasets version -p <dir>` to publish:
#   shards/*.npz + *_manifest.json + materialize_report.json
#   exp5k/best.pt + exp5k/train_report.json
#   gate64/gate_report.json + eval5k/eval_report.json
# Next session: attach that dataset version and re-point --shards/--out at it.